# Müllvision – Training on Google Colab

Runs the same CLI scripts as the lab PC. Before you start: **Runtime → Change runtime type → T4 GPU**.

Layout:
- Images live on Google Drive in `MyDrive/muellvision_data/` (upload the data store there once).
- For speed they are copied to the Colab disk (`/content/data`) at the start of every session.
- Runs (weights, plots) are written to Drive (`MyDrive/muellvision_data/runs`) so they survive disconnects.
- Small result tables (`results/*.csv`) are committed back to the repo (last cell).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Drive and repo
Add a GitHub token with `repo` scope as a Colab secret named `GH_TOKEN` (key icon in the left bar). The repo is private.

In [ ]:
from google.colab import drive, userdata

drive.mount('/content/drive')

import os

DRIVE_DATA = '/content/drive/MyDrive/muellvision_data'
REPO = '/content/muellvision'
BRANCH = 'main'  # or an exp/... branch

token = userdata.get('GH_TOKEN')
if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} https://{token}@github.com/Silverx008/Muellvision.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!pip install -q -r requirements.txt
!pip install -q -e .

## 2. Data to local disk
Reading thousands of small files from Drive during training is slow, so copy once per session.

In [ ]:
!mkdir -p /content/data
!rsync -a --info=progress2 --exclude runs --exclude _lists {DRIVE_DATA}/ /content/data/
os.environ['MV_DATA'] = '/content/data'
os.environ['MV_RUNS'] = f'{DRIVE_DATA}/runs'
!python scripts/build_manifest.py --no-sha1
!python scripts/check_labels.py

### Optional: data prep on Colab
Only needed once; the results are synced back to Drive. Skip if you prepared the data on the laptop.

In [ ]:
# TACO (CC BY 4.0) download + conversion to our 5 classes
# !python scripts/download_taco.py
# !python scripts/convert_taco.py
# Box proposals for the RealWaste/TrashNet folders (review them in Label Studio afterwards!)
# !python scripts/prelabel.py --sources realwaste trashnet --device 0
# !rsync -a /content/data/ {DRIVE_DATA}/ --exclude _lists

## 3. Splits
The committed `splits/v1` is used if it exists. Only create it when it does not (and never after it is frozen).

In [ ]:
if not os.path.exists('splits/v1/fold_0/train.txt'):
    !python scripts/make_splits.py --version v1 --seed 42
!head -20 splits/v1/split_report.md

## 4. Train
Baseline first (YOLO26n, fold 0). `--resume` continues a run after a disconnect.

In [ ]:
!python scripts/train.py --model yolo26n --fold 0 --device 0 --resume

Full cross-validation (4 variants x 5 folds). Finished runs are skipped, so just re-run this cell after a disconnect. A T4 needs roughly 20-60 min per run depending on variant and data size, so spread it over several sessions.

In [ ]:
# !python scripts/run_cv.py --models all --folds all --device 0

## 5. Evaluate

In [ ]:
# !python scripts/eval_cv.py --split v1 --device 0
# !python scripts/tune_tau.py --model yolo26n --split v1 --device 0
# !python scripts/bench.py --weights {os.environ['MV_RUNS']}/detect/v1_yolo26n_f0/weights/best.pt --images /content/data/taco/batch_1/images --device 0 --platform 'Colab T4'
!cat results/experiments.csv

## 6. Save results to the repo
Commits only `results/` to a branch, then open a PR on GitHub.

In [ ]:
import datetime

b = 'exp/colab-' + datetime.datetime.now().strftime('%Y%m%d-%H%M')
!git config user.email 'colab@muellvision.local' && git config user.name 'Colab run'
!git checkout -b {b} && git add results/ && git commit -qm 'Add Colab results' && git push -q origin {b}
print('pushed', b)